# Spotify Track Recommendation System with GNN

Notebook sourced from: https://medium.com/stanford-cs224w/spotify-track-neural-recommender-system-51d266e31e16

The code loads in graph data from the Spotify Million Playlist Dataset and uses Graph Neural Network (GNN) approaches for the automated playlist continuation task; more specifically using convolutional layers from `LightGCN`, `GraphSAGE`, `GAT` to make recommendation predictions.

In [4]:
# General libraries
import json
from pathlib import Path as Data_Path
import os
from os.path import isfile, join
from json import JSONDecodeError
import pickle
import random

import numpy as np
import networkx as nx
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
import matplotlib.pyplot as plt
%matplotlib inline

from tqdm.notebook import tqdm

In [5]:
# Import relevant ML libraries
from typing import Optional, Union

import torch
from torch import Tensor
import torch.nn as nn
from torch.nn import Embedding, ModuleList, Linear
import torch.nn.functional as F

import torch_geometric
import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch.nn.modules.loss import _Loss

from torch_geometric.nn.conv import LGConv, GATConv, SAGEConv
from torch_geometric.typing import Adj, OptTensor, SparseTensor

print(f"Torch version: {torch.__version__}; Torch-cuda version: {torch.version.cuda}; Torch Geometric version: {torch_geometric.__version__}.")

/Users/tiffanykou/Desktop/git/gnn-recommender/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/Users/tiffanykou/Desktop/git/gnn-recommender/.venv/lib/python3.12/site-packages/torch_scatter/_version_cpu.so, 0x0006): Symbol not found: __ZN5torch3jit17parseSchemaOrNameERKNSt3__112basic_stringIcNS1_11char_traitsIcEENS1_9allocatorIcEEEEb
  Referenced from: <C299992C-CA19-3A84-B691-906D7150EEF3> /Users/tiffanykou/Desktop/git/gnn-recommender/.venv/lib/python3.12/site-packages/torch_scatter/_version_cpu.so
  Expected in:     <9CC9ADC2-DF98-3E88-B66B-3D41B7AF7AAF> /Users/tiffanykou/Desktop/git/gnn-recommender/.venv/lib/python3.12/site-packages/torch/lib/libtorch_cpu.dylib
  import torch_geometric.typing
/Users/tiffanykou/Desktop/git/gnn-recommender/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'.

Torch version: 2.2.0; Torch-cuda version: None; Torch Geometric version: 2.7.0.


Set random seed to ensure reproducibility for our work

In [6]:
# set the seed for reproducibility
SEED = 224
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Download Dataset

Since the dataset is not available from download anymore from [aicrowd](https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge), we alternatively downloaded from [here](https://gitlab.ec-lyon.fr/ypelleri/big-data-project/-/tree/main/spotify_million_playlist_dataset/data?ref_type=heads). The curl commands to download sample of the data are as follows:

In [7]:
# # create destination
# mkdir -p spotify_million_playlist/data

# # URL-encode project path (safe on mac)
# ENC_PROJECT=$(python3 - <<'PY'
# import urllib.parse
# print(urllib.parse.quote('ypelleri/big-data-project', safe=''))
# PY
# )

# # optional: set PRIVATE header if repo is private
# AUTH_HEADER=""
# # if private:
# # AUTH_HEADER="-H \"PRIVATE-TOKEN: $GITLAB_TOKEN\""

# # fetch up to 100 filenames and download them
# curl -s $AUTH_HEADER "https://gitlab.ec-lyon.fr/api/v4/projects/${ENC_PROJECT}/repository/tree?path=spotify_million_playlist_dataset%2Fdata&per_page=100" \
#   | jq -r '.[].name' \
#   | head -n 100 \
#   | while read -r fname; do
#       echo "downloading $fname"
#       curl -fL $AUTH_HEADER "https://gitlab.ec-lyon.fr/ypelleri/big-data-project/-/raw/main/spotify_million_playlist_dataset/data/${fname}" \
#         -o "spotify_million_playlist/data/${fname}"
#     done

## Loading and processing data

The data contains a million Spotify playlists created by Spotify between Jan 2010 and Oct 2017 as well as all of their tracks. The full dataset folder is around 34 GB and contains 1,000 JSON files each roughly 34 MB and containing 1,000 playlists.

For this model, we restrict our attention to just the first 200 files (200,000 playlists).

In [8]:
DATA_DIR = 'spotify_million_playlist/data'

In [9]:
with open(f"{DATA_DIR}/{os.listdir(DATA_DIR)[0]}") as jf:
  example_file = json.load(jf)

print(json.dumps(example_file['playlists'][0], indent=4))

{
    "name": "litty titty",
    "collaborative": "false",
    "pid": 115000,
    "modified_at": 1508371200,
    "num_tracks": 212,
    "num_albums": 126,
    "num_followers": 1,
    "tracks": [
        {
            "pos": 0,
            "artist_name": "Travis Scott",
            "track_uri": "spotify:track:0ESJlaM8CE1jRWaNtwSNj8",
            "artist_uri": "spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY",
            "track_name": "beibs in the trap",
            "album_uri": "spotify:album:42WVQWuf1teDysXiOupIZt",
            "duration_ms": 213863,
            "album_name": "Birds In The Trap Sing McKnight"
        },
        {
            "pos": 1,
            "artist_name": "Travis Scott",
            "track_uri": "spotify:track:6gBFPUFcJLzWGx4lenP6h2",
            "artist_uri": "spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY",
            "track_name": "goosebumps",
            "album_uri": "spotify:album:42WVQWuf1teDysXiOupIZt",
            "duration_ms": 243836,
            "album_name": "Birds 

Now let's get ready to load in our data. First, we define a few simple, helfpul classes.

In [10]:
"""
Here we define classes for the data that we are going to load. The data is stored in JSON files, each
which contain playlists, which themselves contain tracks. Thus, we define three classes:
  Track       --> contains information for a specific track (its id, name, etc.)
  Playlist    --> contains information for a specific playlist (its id, name, etc. as well as a list of Tracks)
  JSONFile    --> contains the loaded json file and stores a dictionary of all of the Playlists

Note: if we were to use the artist information, we could make an Artist class
"""

class Track:
  """
  Simple class for a track, containing its attributes:
    1. URI (a unique id)
    2. Name
    3. Artist info (URI and name)
    4. Parent playlist
  """

  def __init__(self, track_dict, playlist):
    self.uri = track_dict["track_uri"]
    self.name = track_dict["track_name"]
    self.artist_uri = track_dict["artist_uri"]
    self.artist_name = track_dict["artist_name"]
    self.playlist = playlist

  def __str__(self):
    return f"Track {self.uri} called {self.name} by {self.artist_uri} ({self.artist_name}) in playlist {self.playlist}."

  def __repr__(self):
    return f"Track {self.uri}"

class Playlist:
  """
  Simple class for a playlist, containing its att≤ributes:
    1. Name (playlist and its associated index)
    2. Title (playlist title in the Spotify dataset)
    3. Loaded dictionary from the raw json for the playlist
    4. Dictionary of tracks (track_uri : Track), populated by .load_tracks()
    5. List of artists uris
  """

  def __init__(self, json_data, index):

    self.name = f"playlist_{index}"
    self.title = json_data["name"]
    self.data = json_data

    self.tracks = {}
    self.artists = []

  def load_tracks(self):
    """ Call this function to load all of the tracks in the json data for the playlist."""

    tracks_list = self.data["tracks"]
    self.tracks = {x["track_uri"] : Track(x, self.name) for x in tracks_list}
    self.artists = [x["artist_uri"] for x in tracks_list]

  def __str__(self):
    return f"Playlist {self.name} with {len(self.tracks)} tracks loaded."

  def __repr__(self):
    return f"Playlist {self.name}"

class JSONFile:
  """
  Simple class for a JSON file, containing its attributes:
    1. File Name
    2. Index to begin numbering playlists at
    3. Loaded dictionary from the raw json for the full file
    4. Dictionary of playlists (name : Playlist), populated by .process_file()
  """

  def __init__(self, data_path, file_name, start_index):

    self.file_name = file_name
    self.start_index = start_index

    with open(join(data_path, file_name)) as json_file:
      json_data = json.load(json_file)
    self.data = json_data

    self.playlists = {}

  def process_file(self):
    """ Call this function to load all of the playlists in the json data."""

    for i, playlist_json in enumerate(self.data["playlists"]):
      playlist = Playlist(playlist_json, self.start_index + i)
      playlist.load_tracks()
      self.playlists[playlist.name] = playlist

  def __str__(self):
    return f"JSON {self.file_name} has {len(self.playlists)} playlists loaded."

  def __repr__(self):
    return self.file_name


Now let's load all 200 json files.

In [11]:
DATA_PATH = Data_Path('spotify_million_playlist/data')
N_FILES_TO_USE = 200

file_names = sorted(os.listdir(DATA_PATH))
file_names_to_use = file_names[:N_FILES_TO_USE]

n_playlists = 0

# load each json file, and store it in a list of files
JSONs = []
for file_name in tqdm(file_names_to_use, desc='Files processed: ', unit='files', total=len(file_names_to_use)):
  try:
    json_file = JSONFile(DATA_PATH, file_name, n_playlists)
    json_file.process_file()
    n_playlists += len(json_file.playlists)
    JSONs.append(json_file)
  except JSONDecodeError as e:
    print(f'Skipping {file_name} due to parsing error: {e}')
    continue

print(f'\n{len(JSONs)} files have been successfully loaded!')

Files processed:   0%|          | 0/200 [00:00<?, ?files/s]

Skipping mpd.slice.161000-161999.json due to parsing error: Unterminated string starting at: line 100010 column 21 (char 4939773)
Skipping mpd.slice.189000-189999.json due to parsing error: Expecting ',' delimiter: line 415074 column 13 (char 20549632)
Skipping mpd.slice.215000-215999.json due to parsing error: Expecting value: line 147390 column 34 (char 7278592)
Skipping mpd.slice.23000-23999.json due to parsing error: Expecting property name enclosed in double quotes: line 279689 column 14 (char 13848576)
Skipping mpd.slice.230000-230999.json due to parsing error: Unterminated string starting at: line 35501 column 35 (char 1744891)
Skipping mpd.slice.231000-231999.json due to parsing error: Unterminated string starting at: line 401864 column 21 (char 19861502)
Skipping mpd.slice.271000-271999.json due to parsing error: Unterminated string starting at: line 171784 column 34 (char 8486909)

193 files have been successfully loaded!


In [12]:
playlist_data = {}
playlists = []
tracks = []

# build list of all unique playlists, tracks
for json_file in tqdm(JSONs):
  playlists += [p.name for p in json_file.playlists.values()]
  tracks += [track.uri for playlist in json_file.playlists.values() for track in list(playlist.tracks.values())]
  playlist_data = playlist_data | json_file.playlists

  0%|          | 0/193 [00:00<?, ?it/s]